In [1]:
import os
import random
from PIL import Image
import numpy as np
from imgaug import augmenters as iaa

# Compatibility for very new numpy versions
try:
    np.bool
except Exception:
    np.bool = np.bool_

# -------------------------
# Helper: find starting index for augmented filenames
# -------------------------
def find_start_index(folder_path, prefix="aug_", ext_tuple=(".jpg", ".png", ".jpeg")):
    max_idx = -1
    for fname in os.listdir(folder_path):
        name, ext = os.path.splitext(fname)
        if not ext.lower() in ext_tuple:
            continue
        if name.startswith(prefix):
            try:
                idx = int(name[len(prefix):])
                if idx > max_idx:
                    max_idx = idx
            except Exception:
                pass
    return max_idx + 1

skin_cancer_augmenter = iaa.Sequential([

    # -----------------------------
    # Geometry (most important)
    # -----------------------------
    iaa.Fliplr(0.5),
    iaa.Flipud(0.5),
    iaa.Affine(
        rotate=(-25, 25),
        scale=(0.85, 1.15),
        shear=(-8, 8),
        translate_percent={"x": (-0.08, 0.08), "y": (-0.08, 0.08)},
        mode="edge"
    ),

    # -----------------------------
    # Illumination (safe)
    # -----------------------------
    iaa.Sometimes(0.7, iaa.Sequential([
        iaa.Multiply((0.9, 1.1)),          # brightness
        iaa.LinearContrast((0.9, 1.1)),    # contrast
    ])),

    # -----------------------------
    # Texture robustness
    # -----------------------------
    iaa.SomeOf((0, 2), [
        iaa.GaussianBlur(sigma=(0.0, 1.0)),
        iaa.Sharpen(alpha=(0.0, 0.2), lightness=(0.9, 1.1)),
        iaa.AdditiveGaussianNoise(scale=(0, 0.005 * 255))
    ], random_order=True),

    # -----------------------------
    # Very mild color shift (optional)
    # -----------------------------
    iaa.Sometimes(0.08, iaa.AddToHueAndSaturation((-4, 4))),

    # -----------------------------
    # Minor occlusion (simulate hair / bubbles)
    # -----------------------------
    iaa.Sometimes(0.06, iaa.CoarseDropout(
        (0.001, 0.008),
        size_percent=(0.004, 0.015)
    ))

], random_order=True)


def get_next_image(available_images, original_images):
    if len(available_images) == 0:
        available_images.extend(original_images)
        random.shuffle(available_images)
    return available_images.pop()

# -------------------------
# Save augmented images
# -------------------------
def save_augmented_images(folder_path, images, augmenter, target_count):
    current_count = len(images)
    to_generate = max(0, target_count - current_count)
    if to_generate == 0:
        return

    # Only ORIGINAL images (never augmented ones)
    original_images = [img for img in images if not os.path.basename(img).startswith("aug_")]
    if len(original_images) == 0:
        print(f"No original images found in {folder_path}. Skipping.")
        return

    # Pool for non-repeating selection
    available_images = original_images.copy()
    random.shuffle(available_images)

    image_index = find_start_index(folder_path, prefix="aug_")

    for _ in range(to_generate):
        try:
            img_path = get_next_image(available_images, original_images)

            img = Image.open(img_path).convert("RGB")
            img_array = np.array(img)

            augmented = augmenter(image=img_array)
            if augmented is None:
                raise RuntimeError("augmenter returned None")

            if augmented.dtype != np.uint8:
                augmented = np.clip(augmented, 0, 255).astype(np.uint8)

            new_filename = os.path.join(folder_path, f"aug_{image_index}.jpg")
            while os.path.exists(new_filename):
                image_index += 1
                new_filename = os.path.join(folder_path, f"aug_{image_index}.jpg")

            Image.fromarray(augmented).save(new_filename, quality=95)
            image_index += 1
            current_count += 1

        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

# -------------------------
# Main augment_data function
# -------------------------
def augment_data(dataset_path, target_count_per_class, augmenter=skin_cancer_augmenter):
    if not os.path.isdir(dataset_path):
        raise ValueError(f"dataset_path not found: {dataset_path}")

    folders = [d for d in sorted(os.listdir(dataset_path)) if os.path.isdir(os.path.join(dataset_path, d))]
    if not folders:
        print("No subfolders found in dataset_path.")
        return

    for folder in folders:
        folder_path = os.path.join(dataset_path, folder)
        images = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        current = len(images)
        if current >= target_count_per_class:
            print(f"'{folder}' already has {current} images. Skipping.")
            continue

        print(f"Augmenting '{folder}': {current} -> {target_count_per_class}")
        save_augmented_images(folder_path, images, augmenter, target_count_per_class)

    print("Data augmentation completed!")


C:\Users\Faysal Ahmmed\AppData\Local\Temp\ipykernel_6476\2845872823.py:9: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  np.bool


In [2]:
augment_data(r"Kaggle 10k+3k+Preprocessed AUG", target_count_per_class = 10000)

Augmenting 'Benign': 7300 -> 10000
Augmenting 'Malignant': 6602 -> 10000
Data augmentation completed!
